# M09 · Tutor inteligente — evaluación ampliada
**Francisco Faubel · Benjamin Ortiz**  
**Pista:** Reglas · **IA declarada:** ChatGPT

Este cuaderno contiene **20 casos oficiales** de la ficha M09 (14 de entrenamiento y 6 de prueba) y **200 casos sintéticos adicionales**, claramente diferenciados. Los casos sintéticos **no tienen resultado real conocido**, así que solo permiten observar predicciones, no medir exactitud ni sobreajuste. El sistema no se vuelve a entrenar ni cambia su regla: **SÍ si aciertos ≥ 70 %; NO en caso contrario**.

Usa exclusivamente los datos proporcionados en la conversación y casos ficticios de práctica.

In [ ]:
import pandas as pd
import numpy as np

UMBRAL = 70

def predecir(aciertos):
    if not 0 <= aciertos <= 100:
        raise ValueError("El porcentaje de aciertos debe estar entre 0 y 100")
    return 1 if aciertos >= UMBRAL else 0

def texto(valor):
    return "SÍ" if valor == 1 else "NO"

print("Regla fija: SÍ si aciertos >= 70%; NO si aciertos < 70%")

## 1. Datos oficiales M09 (no modificar la división)
La columna `real` es la etiqueta de la ficha, y **nunca forma parte de las entradas de `predecir`**.

In [ ]:
oficial = pd.DataFrame([
    # caso, aciertos, pistas, tiempo, temas, real
    (1,51,14,6.1,5,0),(2,58,1,9.6,3,0),(3,67,9,2.8,1,1),
    (4,43,12,10.7,5,0),(5,55,12,6.1,5,0),(6,53,4,9.9,6,0),
    (7,93,1,1.6,1,1),(8,53,12,8.9,5,0),(9,68,10,3.6,2,0),
    (10,98,4,8.6,2,1),(11,38,7,6.9,3,0),(12,21,12,2.0,6,0),
    (13,90,8,2.6,0,1),(14,66,0,10.4,2,0),
    (15,94,5,2.6,0,1),(16,92,13,3.7,4,1),(17,78,2,5.3,3,1),
    (18,42,5,5.8,4,0),(19,67,3,8.3,1,0),(20,83,1,4.7,1,1),
],columns=["caso","aciertos","pistas","tiempo","temas","real"])
oficial["conjunto"] = np.where(oficial.caso <= 14,"Entrenamiento","Prueba")
oficial["prediccion"] = oficial.aciertos.apply(predecir)
oficial["prediccion_texto"] = oficial.prediccion.map(texto)
oficial["tipo"] = np.select([
    (oficial.real == 1) & (oficial.prediccion == 1),
    (oficial.real == 1) & (oficial.prediccion == 0),
    (oficial.real == 0) & (oficial.prediccion == 1),
    (oficial.real == 0) & (oficial.prediccion == 0),
], ["VP","FN","FP","VN"], default="REVISAR")
display(oficial)

In [ ]:
def evaluar(tabla):
    vp = int(((tabla.real == 1) & (tabla.prediccion == 1)).sum())
    fn = int(((tabla.real == 1) & (tabla.prediccion == 0)).sum())
    fp = int(((tabla.real == 0) & (tabla.prediccion == 1)).sum())
    vn = int(((tabla.real == 0) & (tabla.prediccion == 0)).sum())
    assert vp+fn+fp+vn == len(tabla)
    division = lambda a,b: round(100*a/b,2) if b else np.nan
    return {"Casos":len(tabla),"VP":vp,"FN":fn,"FP":fp,"VN":vn,
            "Exactitud (%)":division(vp+vn,len(tabla)),
            "Precisión (%)":division(vp,vp+fp),
            "Sensibilidad (%)":division(vp,vp+fn),
            "Especificidad (%)":division(vn,vn+fp)}

resumen = pd.DataFrame({
    "Entrenamiento": evaluar(oficial[oficial.conjunto == "Entrenamiento"]),
    "Prueba": evaluar(oficial[oficial.conjunto == "Prueba"])
}).T
display(resumen)
assert resumen.loc["Entrenamiento",["VP","FN","FP","VN"]].tolist() == [3,1,0,10]
assert resumen.loc["Prueba",["VP","FN","FP","VN"]].tolist() == [4,0,0,2]
print("Matrices oficiales verificadas.")

## 2. 200 casos nuevos de práctica (simulados, sin etiqueta real)
Se construyen con semilla fija para poder repetir la práctica. Incluyen porcentajes cercanos al umbral y combinaciones extremas de pistas/tiempo. **La salida no demuestra si el sistema acertó.**

In [ ]:
rng = np.random.default_rng(20261103)
aciertos = np.concatenate([
    rng.integers(0,101,size=100),     # variedad general
    rng.integers(60,81,size=80),      # zona difícil alrededor del umbral
    np.array([0,1,20,40,50,60,64,65,66,67,68,69,70,70,71,72,75,85,99,100])
])
assert len(aciertos)==200
simulados = pd.DataFrame({
    "caso": [f"S{i:03d}" for i in range(1,201)],
    "aciertos": aciertos,
    "pistas": rng.integers(0,16,size=200),
    "tiempo": np.round(rng.uniform(1,12,size=200),1),
    "temas": rng.integers(0,7,size=200),
})
simulados["prediccion"] = simulados.aciertos.apply(predecir)
simulados["salida"] = simulados.prediccion.map(texto)
simulados["resultado_real"] = "DESCONOCIDO"
print(f"Generados {len(simulados)} casos sintéticos")
print("Predicción SÍ:", int((simulados.prediccion==1).sum()))
print("Predicción NO:", int((simulados.prediccion==0).sum()))
display(simulados)

## 3. Casos especialmente difíciles (pruebas de estrés)
Hay casos con muchos aciertos pero numerosas pistas, y otros con menos de 70 % pero muy pocas pistas. Esto permite **discutir las limitaciones de la regla**, sin inventar etiquetas reales.

In [ ]:
extremos = simulados[
    ((simulados.aciertos>=85)&(simulados.pistas>=12)) |
    ((simulados.aciertos<70)&(simulados.pistas<=2)) |
    (simulados.aciertos.isin([69,70]))
]
print("Casos de estrés:",len(extremos))
display(extremos)

## 4. Cargar la lista que entregue el profesor
Pegá los porcentajes de aciertos en `porcentajes_del_profesor`. **No cambies la regla**. Si el profesor entrega pistas, tiempo y temas, no afectan la predicción del sistema declarado.

In [ ]:
# REEMPLAZAR esta lista por los porcentajes de los casos nuevos del profesor
porcentajes_del_profesor = [69,70,71,90,45,67,82,100,0,75]

nuevos = pd.DataFrame({"caso":[f"N{i}" for i in range(1,len(porcentajes_del_profesor)+1)],
                       "aciertos":porcentajes_del_profesor})
nuevos["prediccion"] = nuevos.aciertos.apply(predecir).map(texto)
display(nuevos)
print("Estas son predicciones, NO resultados de exactitud: falta la etiqueta real.")

## 5. Opcional: evaluar una lista nueva SOLO si el profesor entrega resultados reales
Pegar los resultados reales **después** de generar las predicciones. Si no los entrega, no inventarlos.

In [ ]:
# Ejemplo de uso (desactivado hasta tener etiquetas reales):
# etiquetas_reales = [0, 1, 1, 0, 0, 1, 1, 1, 0, 1]  # NO INVENTAR: sustituir por datos reales del profesor
# assert len(etiquetas_reales) == len(porcentajes_del_profesor)
# evaluables = pd.DataFrame({"real": etiquetas_reales,
#                            "prediccion": [predecir(v) for v in porcentajes_del_profesor]})
# display(pd.DataFrame([evaluar(evaluables)]))
print("Esta sección espera etiquetas reales proporcionadas por el profesor.")

## 6. Exportación de predicciones sintéticas
Este CSV contiene **solo predicciones y resultados reales desconocidos**. No mezclarlo con el archivo oficial M09.

In [ ]:
simulados.to_csv("M09_200_casos_simulados.csv",index=False)
print("Guardado: M09_200_casos_simulados.csv")